In [1]:
# ===============================
# SPLIT + TRAIN + EVAL
# ===============================

import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from utils import WBCDataset
import config

In [2]:
# ------------------
# CONFIG
# ------------------

DATA_DIR = "./"
TRAIN_CSV = os.path.join(DATA_DIR, "train_metadata.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")

BATCH_SIZE = 128
EPOCHS = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# # ------------------
# # STEP 1: STRATIFIED SPLIT
# # ------------------

# full_df = pd.read_csv(TRAIN_CSV)

# train_df, val_df = train_test_split(
#     full_df,
#     test_size=0.15,
#     stratify=full_df["label"],
#     random_state=42
# )

# train_df.to_csv("train_split.csv", index=False)
# val_df.to_csv("val_split.csv", index=False)

# print("Train size:", len(train_df))
# print("Val size:", len(val_df))


Train size: 24565
Val size: 4336


## VANILLA

In [ ]:
# ------------------
# DATASET
# ------------------
train_dataset = WBCDataset("train_split.csv", TRAIN_IMG_DIR)
val_dataset   = WBCDataset("val_split.csv", TRAIN_IMG_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ------------------
# STEP 2: MODEL
# ------------------

model = models.resnet50(weights="IMAGENET1K_V1")
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)
model = model.to(DEVICE)

# class weights computed ONLY from train split
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.asarray(sorted(train_df["label"].unique())),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ------------------
# STEP 3: TRAIN + VALIDATE
# ------------------

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # ----- VALIDATION -----
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")

# Save tuned model
torch.save(model.state_dict(), "resnet_wbc_finetuned_split.pth")

100%|██████████| 192/192 [02:33<00:00,  1.25it/s]


Epoch 1/15 | Train Loss: 1.3095 | Val Macro-F1: 0.4924


100%|██████████| 192/192 [02:32<00:00,  1.26it/s]


Epoch 2/15 | Train Loss: 0.7065 | Val Macro-F1: 0.4181


100%|██████████| 192/192 [02:32<00:00,  1.26it/s]


...

Epoch 13/15 | Train Loss: 0.0032 | Val Macro-F1: 0.6608


100%|██████████| 192/192 [02:33<00:00,  1.25it/s]


Epoch 14/15 | Train Loss: 0.0024 | Val Macro-F1: 0.6599


100%|██████████| 192/192 [02:32<00:00,  1.26it/s]


Epoch 15/15 | Train Loss: 0.0013 | Val Macro-F1: 0.6590
Feature extraction complete.
Train features: (24565, 2048)
Val features: (4336, 2048)


## DATA AUGMENTATION

In [9]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader

import numpy as np
from tqdm import tqdm

from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

import wandb
import config
from utils import WBCDataset
from datetime import datetime

import os
os.environ["WANDB_START_METHOD"] = "thread"

import wandb
wandb.login(key=config.WANDKEY)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [10]:
wandb.init(
    project="wbc-classification",
    config={
        "model": "resnet50",
        "lr": 1e-4,
        "batch_size": config.BATCH_SIZE,
        "epochs": config.EPOCHS,
        "augmentation": True,
        "weight_decay": 1e-4,
        "scheduler": "cosine_annealing",
        "denoising": "True"
    }
)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [11]:
TRAIN_IMG_DIR = "train"

train_dataset = WBCDataset("train_split.csv", TRAIN_IMG_DIR, augment=True)
val_dataset   = WBCDataset("val_split.csv", TRAIN_IMG_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [12]:
model = models.resnet50(weights="IMAGENET1K_V1")

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(config.CLASS_NAMES))

model = model.to(config.DEVICE)

In [13]:
import pandas as pd

train_df = pd.read_csv("train_split.csv")

classes = np.array(config.CLASS_NAMES)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(config.DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [14]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=wandb.config.lr,
    weight_decay=wandb.config.weight_decay
)

# scheduler = torch.optim.lr_scheduler.ExponentialLR(
#     optimizer,
#     gamma=0.95
# )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.EPOCHS
)

In [15]:
def log_confusion_matrix(all_true, all_preds):
    cm = confusion_matrix(all_true, all_preds)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    fig = plt.figure(figsize=(10,8))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=config.CLASS_NAMES,
                yticklabels=config.CLASS_NAMES)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")

    wandb.log({"confusion_matrix": wandb.Image(fig)})
    plt.show()

    wandb.log({"classification_report": wandb.Table(
        dataframe=pd.DataFrame(
            classification_report(all_true, all_preds, target_names=config.CLASS_NAMES, output_dict=True)
        ).transpose()
    )})

In [ ]:
best_val_f1 = 0.0
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"{config.WEIGHTS_PATH}_{timestamp}.pth"

for epoch in range(wandb.config.epochs):

    # TRAIN
    model.train()
    running_loss = 0

    for batch in tqdm(train_loader):
        images = batch[0].to(config.DEVICE)
        labels = batch[1].to(config.DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    scheduler.step()
    # VALIDATION
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in val_loader:
            images = batch[0].to(config.DEVICE)
            labels = batch[1]

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

    # WANDB LOG
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_f1": val_f1,
        "lr": optimizer.param_groups[0]["lr"]
    })

    # SAVE BEST
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_f1": val_f1
        }, model_path)
        wandb.save(model_path)
        print("New best model saved!!")
        log_confusion_matrix(all_true=all_true, all_preds=all_preds)
        print("confusion matrix and classification report logged!!")
        #artifact = wandb.Artifact("best-model", type="model")
        #artifact.add_file("best_model.pth")
        #wandb.log_artifact(artifact)

print("Best Val F1:", best_val_f1)

 48%|████▊     | 92/192 [02:27<01:36,  1.03it/s]

## Without wandb

In [ ]:
# ===============================
# DATASET WITH AUGMENTATION
# ===============================


# Train dataset gets augmentation
train_dataset = WBCDataset("train_split.csv", TRAIN_IMG_DIR, augment=True)

# Validation dataset does NOT
val_dataset = WBCDataset("val_split.csv", TRAIN_IMG_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# ===============================
# MODEL
# ===============================

model = models.resnet50(weights="IMAGENET1K_V1")
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)
model.load_state_dict(torch.load("resnet_wbc_augmented_best.pth", map_location=DEVICE))
model = model.to(DEVICE)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.asarray(config.CLASS_NAMES),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)


# ===============================
# TRAIN + SAVE BEST MODEL
# ===============================

best_val_f1 = 0.0

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # ----- VALIDATION -----
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "resnet_wbc_augmented_best.pth")
        print("✔ Saved new best model")

print("Best Val Macro-F1:", best_val_f1)

C:\Users\fedem\AppData\Local\Temp\ipykernel_24948\2180166668.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("resnet_wbc_augmented_best

Epoch 1/15 | Train Loss: 0.2102 | Val Macro-F1: 0.6745
✔ Saved new best model


100%|██████████| 192/192 [03:33<00:00,  1.11s/it]


Epoch 2/15 | Train Loss: 0.1637 | Val Macro-F1: 0.6709


100%|██████████| 192/192 [03:25<00:00,  1.07s/it]


...

Epoch 13/15 | Train Loss: 0.0766 | Val Macro-F1: 0.7036
✔ Saved new best model


100%|██████████| 192/192 [18:15<00:00,  5.71s/it]  


Epoch 14/15 | Train Loss: 0.0743 | Val Macro-F1: 0.6626


100%|██████████| 192/192 [03:20<00:00,  1.04s/it]


Epoch 15/15 | Train Loss: 0.0688 | Val Macro-F1: 0.7124
✔ Saved new best model
Best Val Macro-F1: 0.7124219057491199


## MULTI STAGE FINETUNING

In [1]:
TRAIN_IMG_DIR = "train"

import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from utils import WBCDataset
import config

In [2]:
train_dataset = WBCDataset("train_split.csv", TRAIN_IMG_DIR, augment=True)
val_dataset   = WBCDataset("val_split.csv", TRAIN_IMG_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [16]:
# ===============================
# 🔥 FULL TRAINING PIPELINE (W&B + MULTI-STAGE + COSINE + BEST CM)
# ===============================

import torch
import torch.nn as nn
import numpy as np
from torchvision import models
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import wandb
import config
import pandas as pd

train_df = pd.read_csv("train_split.csv")

# -------------------------------
# CONFIG
# -------------------------------
wandb.init(
    project="wbc-classification",
    config={
        "model": "resnet50",
        "num_classes": 13,
        "stage1_epochs": 8,
        "stage2_epochs": 35,
        "lr_head": 1e-3,
        "lr_backbone_low": 1e-5,
        "lr_backbone_high": 5e-6,
        "lr_fc": 1e-4,
        "weight_decay": 1e-4,
        "DEVICE":config.DEVICE
    }
)

config = wandb.config

# -------------------------------
# MODEL
# -------------------------------
model = models.resnet50(weights="IMAGENET1K_V1")
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, config.num_classes)
model = model.to(config.DEVICE)

# -------------------------------
# LOSS
# -------------------------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.asarray(sorted(train_df["label"].unique())),
    y=train_df["label"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(config.DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------------------
# W&B TRACK
# -------------------------------
wandb.watch(model, log="all", log_freq=100)

# -------------------------------
# EVAL FUNCTION
# -------------------------------
def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(config.DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")
    return val_f1, all_preds, all_true


best_val_f1 = 0.0
best_preds, best_true = None, None

# ============================================================
# ==================== STAGE 1 ===============================
# ============================================================
print("\n========== STAGE 1 ==========")

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.fc.parameters(), lr=config.lr_head)

for epoch in range(config.stage1_epochs):

    model.train()
    running_loss = 0

    for images, labels, _ in tqdm(train_loader):
        images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_f1, _, _ = evaluate(model, val_loader)

    wandb.log({
        "stage": 1,
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_f1": val_f1
    })

    print(f"[Stage 1] Epoch {epoch+1} | Val F1: {val_f1:.4f}")


# ============================================================
# ==================== STAGE 2 ===============================
# ============================================================
print("\n========== STAGE 2 ==========")

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.layer1.parameters(), "lr": config.lr_backbone_low},
    {"params": model.layer2.parameters(), "lr": config.lr_backbone_low},
    {"params": model.layer3.parameters(), "lr": config.lr_backbone_high},
    {"params": model.layer4.parameters(), "lr": config.lr_backbone_high},
    {"params": model.fc.parameters(), "lr": config.lr_fc},
], weight_decay=config.weight_decay)

# 🔥 COSINE SCHEDULER
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.stage2_epochs
)

for epoch in range(config.stage2_epochs):

    model.train()
    running_loss = 0

    for images, labels, _ in tqdm(train_loader):
        images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_f1, preds, true = evaluate(model, val_loader)

    # step scheduler AFTER epoch
    scheduler.step()

    wandb.log({
        "stage": 2,
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_f1": val_f1,
        "lr": scheduler.get_last_lr()[0]
    })

    print(f"[Stage 2] Epoch {epoch+1} | Val F1: {val_f1:.4f}")

    # -------------------------------
    # SAVE BEST MODEL + STORE CM DATA
    # -------------------------------
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_preds = preds
        best_true = true

        save_path = "best_model.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1": val_f1,
            "epoch": epoch,
        }, save_path)

        artifact = wandb.Artifact("resnet50-wbc", type="model")
        artifact.add_file(save_path)
        wandb.log_artifact(artifact)

        print("✔ Saved new best model")


# ============================================================
# 📊 LOG CONFUSION MATRIX OF BEST MODEL
# ============================================================
print("\n📊 Logging best confusion matrix...")

wandb.log({
    "best_confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=best_true,
        preds=best_preds
    ),
    "best_val_f1": best_val_f1
})

print("\n🔥 Best Val Macro-F1:", best_val_f1)

wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.



========== STAGE 1 ==========


100%|██████████| 192/192 [00:50<00:00,  3.78it/s]


[Stage 1] Epoch 1 | Val F1: 0.2659


100%|██████████| 192/192 [00:50<00:00,  3.78it/s]


...

100%|██████████| 192/192 [02:09<00:00,  1.49it/s]


[Stage 2] Epoch 34 | Val F1: 0.6110


100%|██████████| 192/192 [02:09<00:00,  1.48it/s]


[Stage 2] Epoch 35 | Val F1: 0.6105

📊 Logging best confusion matrix...

🔥 Best Val Macro-F1: 0.6221807982075548


best_val_f1,▁
epoch,▁▁▁▂▂▂▂▂▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
lr,██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
stage,▁▁▁▁▁▁▁▁████████████████████████████████
train_loss,█▆▆▆▅▅▅▅▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_f1,▁▁▂▁▂▃▁▃▄▅▅▆▆▇▇▇▇▇▇▇▇▇█▇███▇██▇█████████
best_val_f1,0.62218
epoch,34
lr,0
stage,2
train_loss,0.23317


In [1]:
# ===============================
# 🔥 FULL PIPELINE (IMBALANCE FIXED)
# ===============================

import torch
import torch.nn as nn
import numpy as np
from torchvision import models
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
import wandb
import pandas as pd
import config

from utils import WBCDataset

# -------------------------------
# CONFIG
# -------------------------------
wandb.init(
    project="wbc-classification",
    config={
        "model": "resnet50",
        "num_classes": 13,
        "stage1_epochs": 8,
        "stage2_epochs": 35,
        "lr_head": 1e-3,
        "lr_backbone_low": 1e-5,
        "lr_backbone_high": 5e-6,
        "lr_fc": 1e-4,
        "weight_decay": 1e-4,
        "batch_size": config.BATCH_SIZE,
        "gamma_focal": 2.0
    }
)

cfg = wandb.config
DEVICE = config.DEVICE

# -------------------------------
# DATASETS
# -------------------------------
train_dataset = WBCDataset("train_split.csv", config.TRAIN_IMG_DIR, augment=True)
val_dataset   = WBCDataset("val_split.csv", config.TRAIN_IMG_DIR, augment=False)

train_df = pd.read_csv("train_split.csv")

# -------------------------------
# 🔥 WEIGHTED SAMPLER
# -------------------------------
class_counts = train_df["label"].value_counts().sort_index()
class_weights = 1.0 / class_counts

sample_weights = train_df["label"].map(class_weights).values
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    sampler=sampler,   # 🔥 key change
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# -------------------------------
# MODEL
# -------------------------------
model = models.resnet50(weights="IMAGENET1K_V1")
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, cfg.num_classes)
model = model.to(DEVICE)

# -------------------------------
# 🔥 FOCAL LOSS
# -------------------------------
class_weights_tensor = torch.tensor(
    class_weights.values, dtype=torch.float
).to(DEVICE)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha)

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma) * ce_loss

criterion = FocalLoss(alpha=class_weights_tensor, gamma=cfg.gamma_focal)

# -------------------------------
# W&B TRACK
# -------------------------------
wandb.watch(model, log="all", log_freq=100)

# -------------------------------
# EVAL FUNCTION
# -------------------------------
def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")
    return val_f1, all_preds, all_true


best_val_f1 = 0.0
best_preds, best_true = None, None

# ============================================================
# ==================== STAGE 1 ===============================
# ============================================================
print("\n========== STAGE 1 ==========")

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.fc.parameters(), lr=cfg.lr_head)

for epoch in range(cfg.stage1_epochs):

    model.train()
    running_loss = 0

    for images, labels, _ in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_f1, _, _ = evaluate(model, val_loader)

    wandb.log({
        "stage": 1,
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_f1": val_f1
    })

    print(f"[Stage 1] Epoch {epoch+1} | Val F1: {val_f1:.4f}")


# ============================================================
# ==================== STAGE 2 ===============================
# ============================================================
print("\n========== STAGE 2 ==========")

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.layer1.parameters(), "lr": cfg.lr_backbone_low},
    {"params": model.layer2.parameters(), "lr": cfg.lr_backbone_low},
    {"params": model.layer3.parameters(), "lr": cfg.lr_backbone_high},
    {"params": model.layer4.parameters(), "lr": cfg.lr_backbone_high},
    {"params": model.fc.parameters(), "lr": cfg.lr_fc},
], weight_decay=cfg.weight_decay)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.stage2_epochs
)

for epoch in range(cfg.stage2_epochs):

    model.train()
    running_loss = 0

    for images, labels, _ in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_f1, preds, true = evaluate(model, val_loader)

    scheduler.step()

    wandb.log({
        "stage": 2,
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_f1": val_f1,
        "lr": scheduler.get_last_lr()[0]
    })

    print(f"[Stage 2] Epoch {epoch+1} | Val F1: {val_f1:.4f}")

    # SAVE BEST
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_preds = preds
        best_true = true

        torch.save(model.state_dict(), "best_model.pth")

        artifact = wandb.Artifact("resnet50-wbc", type="model")
        artifact.add_file("best_model.pth")
        wandb.log_artifact(artifact)

        print("✔ Saved new best model")


# ============================================================
# CONFUSION MATRIX
# ============================================================
wandb.log({
    "best_confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=best_true,
        preds=best_preds
    ),
    "best_val_f1": best_val_f1
})

print("\n🔥 Best Val Macro-F1:", best_val_f1)

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\fedem\_netrc.
wandb: Currently logged in as: fedemezuga3 (fedemezuga3-t-l-com-paris) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



========== STAGE 1 ==========


100%|██████████| 192/192 [00:52<00:00,  3.66it/s]


[Stage 1] Epoch 1 | Val F1: 0.0429


100%|██████████| 192/192 [00:50<00:00,  3.79it/s]


...

100%|██████████| 192/192 [02:06<00:00,  1.52it/s]


[Stage 2] Epoch 35 | Val F1: 0.3326


wandb: ERROR The nbformat package was not found. It is required to save notebook history.



🔥 Best Val Macro-F1: 0.3346742207874082


best_val_f1,▁
epoch,▁▁▁▂▂▂▂▂▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
lr,██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
stage,▁▁▁▁▁▁▁▁████████████████████████████████
train_loss,█▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_f1,▁▂▂▂▂▂▂▂▃▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█▇███████████
best_val_f1,0.33467
epoch,34
lr,0
stage,2
train_loss,5e-05


In [ ]:
# ===============================
# MODEL
# ===============================

model = models.resnet50(weights="IMAGENET1K_V1")
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)
model = model.to(DEVICE)

# Class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.asarray(sorted(train_df["label"].unique())),
    y=train_df["label"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_val_f1 = 0.0


# ============================================================
# ==================== STAGE 1 ===============================
# Train ONLY classifier head
# ============================================================

print("\n========== STAGE 1: Train classifier head ==========")

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)

for epoch in range(5):  # short warmup stage

    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"[Stage 1] Epoch {epoch+1} | "
          f"Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")


# ============================================================
# ==================== STAGE 2 ===============================
# Fine-tune entire network
# ============================================================

print("\n========== STAGE 2: Fine-tune full model ==========")

# Unfreeze everything
for param in model.parameters():
    param.requires_grad = True

# Differential learning rates
optimizer = torch.optim.AdamW([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 5e-6},
    {"params": model.layer4.parameters(), "lr": 5e-6},
    {"params": model.fc.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # ----- VALIDATION -----
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"[Stage 2] Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "resnet_wbc_augmented_best.pth")
        print("✔ Saved new best model")

print("Best Val Macro-F1:", best_val_f1)


========== STAGE 1: Train classifier head ==========


100%|██████████| 192/192 [03:48<00:00,  1.19s/it]


[Stage 1] Epoch 1 | Loss: 2.2141 | Val Macro-F1: 0.2494


100%|██████████| 192/192 [02:09<00:00,  1.48it/s]


[Stage 1] Epoch 2 | Loss: 1.7943 | Val Macro-F1: 0.3273


...

[Stage 2] Epoch 18/20 | Train Loss: 0.2925 | Val Macro-F1: 0.6404
✔ Saved new best model


100%|██████████| 192/192 [03:34<00:00,  1.12s/it]


[Stage 2] Epoch 19/20 | Train Loss: 0.2816 | Val Macro-F1: 0.6424
✔ Saved new best model


100%|██████████| 192/192 [03:29<00:00,  1.09s/it]


[Stage 2] Epoch 20/20 | Train Loss: 0.2708 | Val Macro-F1: 0.6348
Best Val Macro-F1: 0.6424434582612123


In [ ]:
# ============================================
# RESUME STAGE 2 TRAINING
# ============================================

print("\n========== RESUMING STAGE 2 ==========")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "checkpoint.pth"
ADDITIONAL_EPOCHS = 10

# -----------------------
# Rebuild model
# -----------------------

model = models.resnet50(weights=None)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint["model"])
model = model.to(DEVICE)

# -----------------------
# Unfreeze everything
# -----------------------

for param in model.parameters():
    param.requires_grad = True

# -----------------------
# Recreate optimizer (must match original)
# -----------------------

optimizer = torch.optim.AdamW([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 5e-6},
    {"params": model.layer4.parameters(), "lr": 5e-6},
    {"params": model.fc.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

optimizer.load_state_dict(checkpoint["optimizer"])

start_epoch = checkpoint["epoch"] + 1
best_val_f1 = checkpoint.get("best_val_f1", 0.0)

print(f"Resuming from epoch {start_epoch}")
print(f"Previous best Val Macro-F1: {best_val_f1:.4f}")

# -----------------------
# Recreate scheduler
# -----------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=ADDITIONAL_EPOCHS
)

# ============================================
# Continue Training
# ============================================

for epoch in range(start_epoch, start_epoch + ADDITIONAL_EPOCHS):

    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    # ----- VALIDATION -----
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.numpy())

    val_f1 = f1_score(all_true, all_preds, average="macro")

    print(f"[Resume Stage 2] Epoch {epoch} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Macro-F1: {val_f1:.4f}")

    # Save only if improved
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_f1": best_val_f1
        }, CHECKPOINT_PATH)

        print("✔ Saved improved checkpoint")

print("\nBest resumed Val Macro-F1:", best_val_f1)


========== RESUMING STAGE 2 ==========


C:\Users\fedem\AppData\Local\Temp\ipykernel_24948\1885927015.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVIC

Resuming from epoch 10
Previous best Val Macro-F1: 0.0000


100%|██████████| 192/192 [03:29<00:00,  1.09s/it]


[Resume Stage 2] Epoch 10 | Train Loss: 0.1682 | Val Macro-F1: 0.6519
✔ Saved improved checkpoint


...

[Resume Stage 2] Epoch 17 | Train Loss: 0.1718 | Val Macro-F1: 0.6499


100%|██████████| 192/192 [03:37<00:00,  1.13s/it]


[Resume Stage 2] Epoch 18 | Train Loss: 0.1643 | Val Macro-F1: 0.6530


100%|██████████| 192/192 [03:28<00:00,  1.08s/it]


[Resume Stage 2] Epoch 19 | Train Loss: 0.1663 | Val Macro-F1: 0.6509

Best resumed Val Macro-F1: 0.6584929738494291
